# Simple teaching demo: apply response (forward) and recover with `remove_response()`

This notebook is intentionally **minimal** and **self-contained** (aside from requiring that you already computed `resp_combined` and `inv_combined`
in a previous notebook cell). It shows:

1) A synthetic **ground velocity** (Ricker wavelet) with peak ≈ **1e-6 m/s**  
2) A synthetic **digitizer trace in counts** produced by applying the **combined Trillium+Centaur response**  
3) Recovery of velocity with **ObsPy `Trace.remove_response()`**  
4) Time series + spectra + simple amplitude checks

### Key detail (fixes the scaling mess)

`resp_combined.get_evalresp_response(output="VEL")` gives a transfer function **counts → velocity**:

\[ V(f) = G_{C\to V}(f)\,C(f) \]

So for forward simulation (velocity → counts):

\[ C(f) = \frac{V(f)}{G_{C\to V}(f)} \]

We do that in the frequency domain with a small waterlevel for stability.


In [ ]:
from obspy.clients.nrl import NRL

nrl = NRL()

sensor_keys = ["Nanometrics", "Trillium Compact 120 (Vault, Posthole, OBS)", "754 V/m/s"]
datalogger_keys = ["Nanometrics", "Centaur", "40 Vpp (1)", "Off", "Linear phase", "100"]

resp_combined = nrl.get_response(datalogger_keys=datalogger_keys, sensor_keys=sensor_keys)
resp_sensor, _ = nrl._get_response("sensors", keys=sensor_keys)
resp_digitizer, _ = nrl._get_response("dataloggers", keys=datalogger_keys)
print("Combined response:", resp_combined)

In [ ]:
from obspy.core.inventory import Inventory, Network, Station, Channel, Site, Response
from obspy import UTCDateTime

def response_to_inventory(resp: Response, net="XX", sta="NRL1", loc="00", cha="HHZ", sr=100.0) -> Inventory:
    if not isinstance(resp, Response):
        raise TypeError(f"response_to_inventory expected obspy Response, got {type(resp)!r}")

    neto = Network(code=net)
    stao = Station(
        code=sta,
        latitude=0, longitude=0, elevation=0,
        creation_date=UTCDateTime(2020, 1, 1),
        site=Site(name="NRL"),
    )
    chao = Channel(
        code=cha,
        location_code=loc,
        latitude=0, longitude=0,
        elevation=0, depth=0,
        azimuth=0, dip=-90,
        sample_rate=sr,
    )
    chao.response = resp
    stao.channels.append(chao)
    neto.stations.append(stao)

    return Inventory(networks=[neto], source="NRL via ObsPy")

inv_sensor    = response_to_inventory(resp_sensor,    sta="SENSOR",   sr=100.0)
inv_digitizer = response_to_inventory(resp_digitizer, sta="DIGI",     sr=100.0)
inv_combined  = response_to_inventory(resp_combined,  sta="COMBINED", sr=100.0)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from obspy import Trace, UTCDateTime
from obspy.core.inventory import Inventory

# Expect these from earlier NRL cells in your session/notebook
if 'resp_combined' not in globals():
    raise RuntimeError("Missing `resp_combined` (ObsPy Response). Run your NRL get_response() cells first.")
if 'inv_combined' not in globals():
    raise RuntimeError("Missing `inv_combined` (Inventory). Run your response_to_inventory(resp_combined) cell first.")
assert isinstance(inv_combined, Inventory)

def ricker(t, f0):
    a = (np.pi * f0 * t)**2
    return (1.0 - 2.0*a) * np.exp(-a)

def amp_spec(x, dt):
    X = np.fft.rfft(x)
    f = np.fft.rfftfreq(len(x), d=dt)
    return f, np.abs(X)

def evalresp_counts_to_vel(resp, dt, nfft, output="VEL"):
    # returns complex G(f) such that (for output='VEL') V = G * C
    try:
        freqs, G = resp.get_evalresp_response(t_samp=dt, nfft=nfft, output=output)
    except TypeError:
        freqs, G = resp.get_evalresp_response(t_samp=dt, nfft=nfft)
    G = np.asarray(G)
    f = np.fft.rfftfreq(nfft, d=dt)
    if G.shape[0] == nfft:
        G = G[:len(f)]
    if freqs is not None:
        freqs = np.asarray(freqs)
        if freqs.shape[0] == nfft:
            freqs = freqs[:len(f)]
        if freqs.shape == f.shape:
            f = freqs
    return f, G


## 1) True ground motion (velocity)

In [ ]:
sr = 100.0
dt = 1.0/sr
n = 4096
t = (np.arange(n) - n//2) * dt

f0 = 5.0
x = ricker(t, f0)

# scale to peak 1e-6 m/s
x *= 1e-6 / np.max(np.abs(x))

# tiny noise (optional)
rng = np.random.default_rng(0)
x += 1e-10 * rng.standard_normal(n)

tr_true = Trace(x.astype(np.float64))
tr_true.stats.starttime = UTCDateTime(2020, 1, 1)
tr_true.stats.sampling_rate = sr
tr_true.stats.network = "XX"
tr_true.stats.station = "COMBINED"
tr_true.stats.location = "00"
tr_true.stats.channel = "HHZ"

print("True peak (m/s):", np.max(np.abs(tr_true.data)))

plt.figure()
plt.plot(t, tr_true.data)
plt.xlim(-2, 2)
plt.xlabel("Time (s)")
plt.ylabel("Velocity (m/s)")
plt.title("True ground velocity (Ricker), peak ≈ 1e-6 m/s")
plt.show()


## 2) Forward simulation: velocity → counts using the *same* evalresp convention as `remove_response()`

In [ ]:
nfft = n  # keep simple for teaching
f, G_CtoV = evalresp_counts_to_vel(resp_combined, dt=dt, nfft=nfft, output="VEL")  # V = G * C

V = np.fft.rfft(tr_true.data, n=nfft)

# Evaluate forward response: velocity -> counts
f, H = evalresp_counts_to_vel(resp_combined, dt=dt, nfft=nfft, output="VEL")
# despite the helper name, in practice this is usually COUNTS per (m/s) when output="VEL"

V = np.fft.rfft(tr_true.data, n=nfft)
C = H * V                          # <-- multiply, don't divide
counts = np.fft.irfft(C, n=nfft)[:n].astype(np.float64)

tr_counts = tr_true.copy()
tr_counts.data = counts
tr_counts.attach_response(inv_combined)

ratio = np.max(np.abs(tr_counts.data)) / np.max(np.abs(tr_true.data))
print(f"Peak(|counts|)/Peak(|m/s|) ≈ {ratio:.3e}")

plt.figure()
plt.plot(t, tr_counts.data)
plt.xlim(-2, 2)
plt.xlabel("Time (s)")
plt.ylabel("Counts")
plt.title("Synthetic observed seismogram (counts)")
plt.show()


In [ ]:
# Step 2: forward simulate velocity -> counts using evalresp + regularized inverse

nfft = n  # keep teaching-simple
f, G = evalresp_counts_to_vel(resp_combined, dt=dt, nfft=nfft, output="VEL")  # G = V / C (m/s per count)

V = np.fft.rfft(tr_true.data, n=nfft)

# Regularized inverse of G: 1/G ≈ conj(G) / max(|G|, wl)^2
mag = np.abs(G)
mag_max = np.max(mag[1:])  # skip DC for sanity
water_level_db = 60  # like remove_response water_level (dB)
wl = mag_max * 10**(-water_level_db/20)

invG = np.conj(G) / np.maximum(mag, wl)**2

# Also kill DC explicitly (prevents silly offsets)
invG[0] = 0.0# + 0.0j

C = V * invG
counts = np.fft.irfft(C, n=nfft)[:n].astype(np.float64)

tr_counts = tr_true.copy()
tr_counts.data = counts
tr_counts.attach_response(inv_combined)

ratio = np.max(np.abs(tr_counts.data)) / np.max(np.abs(tr_true.data))
print(f"Peak(|counts|)/Peak(|m/s|) ≈ {ratio:.3e} counts per (m/s)")
print("Expected order ~3e8 => 1e-6 m/s -> ~300 counts peak")

plt.figure()
plt.plot(t, tr_counts.data)
plt.xlim(-2, 2)
plt.xlabel("Time (s)")
plt.ylabel("Counts")
plt.title("Synthetic observed seismogram (counts)")
plt.show()

In [ ]:
print(resp_combined.instrument_sensitivity)
if resp_combined.instrument_sensitivity:
    print("Sensitivity value:", resp_combined.instrument_sensitivity.value)
    print("Input units:", resp_combined.instrument_sensitivity.input_units)
    print("Output units:", resp_combined.instrument_sensitivity.output_units)

In [ ]:
# Step 2 (fixed): forward simulate velocity -> counts, with correct scaling

nfft = n
f = np.fft.rfftfreq(nfft, d=dt)

# evalresp: counts -> velocity (m/s per count)
freqs, G_c2v = resp_combined.get_evalresp_response(t_samp=dt, nfft=nfft, output="VEL")
G_c2v = np.asarray(G_c2v)
if G_c2v.shape[0] == nfft:  # sometimes full FFT length
    G_c2v = G_c2v[:len(f)]

# invert: velocity -> counts (counts per m/s)
mag = np.abs(G_c2v)
wl = np.max(mag[1:]) * 10**(-60/20)  # 60 dB waterlevel for stability
H_v2c = np.conj(G_c2v) / np.maximum(mag, wl)**2
H_v2c[0] = 0.0  # DC -> 0 counts for a wavelet

# --- FORCE the correct absolute scaling using known sensitivity at 1 Hz ---
sens = float(resp_combined.instrument_sensitivity.value)  # counts per (m/s)
f_ref = float(resp_combined.instrument_sensitivity.frequency)  # usually 1.0 Hz

k = np.argmin(np.abs(f - f_ref))
scale = sens / np.abs(H_v2c[k])
H_v2c *= scale

# Apply response (frequency domain)
V = np.fft.rfft(tr_true.data, n=nfft)
C = V * H_v2c
counts = np.fft.irfft(C, n=nfft)[:n].astype(np.float64)

tr_counts = tr_true.copy()
tr_counts.data = counts
tr_counts.attach_response(inv_combined)

ratio = np.max(np.abs(tr_counts.data)) / np.max(np.abs(tr_true.data))
print(f"Peak(|counts|)/Peak(|m/s|) ≈ {ratio:.3e} counts per (m/s)")
print("Expected ≈", sens, "counts per (m/s)")

plt.figure()
plt.plot(t, tr_counts.data)
plt.xlim(-2, 2)
plt.xlabel("Time (s)")
plt.ylabel("Counts")
plt.title("Synthetic observed seismogram (counts) — correctly scaled")
plt.show()

In [ ]:
nfft = n
f = np.fft.rfftfreq(nfft, d=dt)

# evalresp: counts -> velocity  (V = G * C), so G has units m/s per count
freqs, G_c2v = resp_combined.get_evalresp_response(t_samp=dt, nfft=nfft, output="VEL")
G_c2v = np.asarray(G_c2v)
if G_c2v.shape[0] == nfft:
    G_c2v = G_c2v[:len(f)]
print("G_c2v units (m/s per count):", G_c2v, resp_combined.instrument_sensitivity.output_units, "/", resp_combined.instrument_sensitivity.input_units)

# forward: velocity -> counts is the exact reciprocal
H_v2c = 1.0 / G_c2v

# DC is often singular because of zeros at origin; just zero it
H_v2c[0] = 0.0

V = np.fft.rfft(tr_true.data, n=nfft)
C = V * H_v2c
counts = np.fft.irfft(C, n=nfft)[:n].astype(np.float64)

tr_counts = tr_true.copy()
tr_counts.data = counts
tr_counts.attach_response(inv_combined)

ratio = np.max(np.abs(tr_counts.data)) / np.max(np.abs(tr_true.data))
print(f"Peak(|counts|)/Peak(|m/s|) ≈ {ratio:.3e}")
print("Instrument sensitivity (counts per m/s):", resp_combined.instrument_sensitivity.value)

## 3) Spectra: true velocity vs observed counts

In [ ]:
f_true, A_true = amp_spec(tr_true.data, dt)
f_obs,  A_obs  = amp_spec(tr_counts.data, dt)

plt.figure()
plt.loglog(f_true[1:], A_true[1:], label="|X_true| (m/s)")
plt.loglog(f_obs[1:],  A_obs[1:],  label="|X_obs| (counts)")
plt.xlabel("Frequency (Hz)")
plt.ylabel("Amplitude")
plt.title("Spectra: true velocity vs observed counts")
plt.legend()
plt.show()


## 4) Deconvolution: recover velocity with `remove_response()`

In [ ]:
tr_rec = tr_counts.copy()

pre_filt = (0.05, 0.1, 40.0, 45.0)
tr_rec.remove_response(
    inventory=inv_combined,
    output="VEL",
    pre_filt=pre_filt,
    water_level=60,
    plot=False
)

print("Recovered peak (m/s):", np.max(np.abs(tr_rec.data)))
print("Peak ratio recovered/true:", np.max(np.abs(tr_rec.data)) / np.max(np.abs(tr_true.data)))

plt.figure()
plt.plot(t, tr_true.data, label="true", alpha=0.9)
plt.plot(t, tr_rec.data,  label="recovered", alpha=0.8)
plt.xlim(-2, 2)
plt.xlabel("Time (s)")
plt.ylabel("Velocity (m/s)")
plt.title("Recovered vs true velocity (expect close match in-band)")
plt.legend()
plt.show()


## 5) Spectra after recovery

In [ ]:
f_rec, A_rec = amp_spec(tr_rec.data, dt)

plt.figure()
plt.loglog(f_true[1:], A_true[1:], label="|X_true| (m/s)")
plt.loglog(f_rec[1:],  A_rec[1:],  label="|X_rec| (m/s)")
plt.xlabel("Frequency (Hz)")
plt.ylabel("Amplitude")
plt.title("Spectra: recovered vs true")
plt.legend()
plt.show()
